In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv("failure_windows.csv")
df['date'] = pd.to_datetime(df['date'])
features = [5,187,188,198,197,241,12,7,4,193,190]
features_cols = [f"smart_{f}_raw" for f in features]
meta_cols = ["date","failure","serial_number"]
print("使用特征列：", features_cols)
df = df[meta_cols + features_cols].fillna(0)

In [ ]:
# ===== 特征提取函数 =====
def extract_features(group):
    group = group.sort_values("date")
    result = {}
    for col in features_cols:
        values = group[col]
        if len(values) == 0:
            # 如果这一列全空，填默认值
            result[f"{col}_max"] = np.nan
            result[f"{col}_mean"] = np.nan
            result[f"{col}_cv"] = np.nan
            continue
            
        # =====计算max、mean、cv=======
        mean = values.mean()
        std = values.std()
        result[f"{col}_max"] = values.max()
        result[f"{col}_mean"] = mean
        result[f"{col}_cv"] = std / mean if mean != 0 else 0
        
        # =======计算diff_mean、diff_max、diff_std=====
        diff = values.diff().dropna()   # 一阶差分
        if len(diff) > 0:
            result[f"{col}_diff_mean"] = diff.mean()
            result[f"{col}_diff_max"] = diff.max()
            result[f"{col}_diff_std"] = diff.std()
        else:
            result[f"{col}_diff_mean"] = 0
            result[f"{col}_diff_max"] = 0
            result[f"{col}_diff_std"] = 0

        # ===== 线性趋势斜率slope=====
        if len(values) >= 2:
            slope = (values.iloc[-1] - values.iloc[0]) / (len(values) - 1)
        else:
            slope = 0
        result[f"{col}_slope"] = slope

        # ===== 偏度（skewness）=====
        skew = values.skew() if len(values) > 2 else 0
        result[f"{col}_skew"] = skew

        # ===== 峰度（kurtosis）=====
        kurt = values.kurt() if len(values) > 3 else 0
        result[f"{col}_kurt"] = kurt
        
        #========最后一天的特征=======
        result[f"{col}_last"] = values.iloc[-1]
        
    # 标签
    result['failure'] = 1
    # 标识
    result['serial_number'] = group['serial_number'].iloc[0]
    return pd.Series(result)

In [ ]:
df = df.sort_values(by=["serial_number", "date"]) # 按日期排序
feature_df = df.groupby('serial_number').apply(extract_features).reset_index(drop=True)
feature_df.to_csv("failure_features.csv", index=False)

In [1]:
## 计算正常样本的特征
import pandas as pd
import numpy as np
df = pd.read_csv("normal_windows.csv")
df['date'] = pd.to_datetime(df['date'])
features = [5,187,188,198,197,241,12,7,4,193,190]
features_cols = [f"smart_{f}_raw" for f in features]
meta_cols = ["date","failure","serial_number"]
print("使用特征列：", features_cols)
df = df[meta_cols + features_cols].fillna(0)

# ===== 特征提取函数 =====
def extract_features(group):
    group = group.sort_values("date")
    result = {}
    for col in features_cols:
        values = group[col]
        if len(values) == 0:
            # 如果这一列全空，填默认值
            result[f"{col}_max"] = np.nan
            result[f"{col}_mean"] = np.nan
            result[f"{col}_cv"] = np.nan
            continue
            
        # =====计算max、mean、cv=======
        mean = values.mean()
        std = values.std()
        result[f"{col}_max"] = values.max()
        result[f"{col}_mean"] = mean
        result[f"{col}_cv"] = std / mean if mean != 0 else 0
        
        # =======计算diff_mean、diff_max、diff_std=====
        diff = values.diff().dropna()   # 一阶差分
        if len(diff) > 0:
            result[f"{col}_diff_mean"] = diff.mean()
            result[f"{col}_diff_max"] = diff.max()
            result[f"{col}_diff_std"] = diff.std()
        else:
            result[f"{col}_diff_mean"] = 0
            result[f"{col}_diff_max"] = 0
            result[f"{col}_diff_std"] = 0

        # ===== 线性趋势斜率slope=====
        if len(values) >= 2:
            slope = (values.iloc[-1] - values.iloc[0]) / (len(values) - 1)
        else:
            slope = 0
        result[f"{col}_slope"] = slope

        # ===== 偏度（skewness）=====
        skew = values.skew() if len(values) > 2 else 0
        result[f"{col}_skew"] = skew

        # ===== 峰度（kurtosis）=====
        kurt = values.kurt() if len(values) > 3 else 0
        result[f"{col}_kurt"] = kurt
        
        #========最后一天的特征=======
        result[f"{col}_last"] = values.iloc[-1]
        
    # 标签
    result['failure'] = 0
    # 标识
    result['serial_number'] = group['serial_number'].iloc[0]
    return pd.Series(result)

df = df.sort_values(by=["serial_number", "date"]) # 按日期排序
feature_df = df.groupby('serial_number').apply(extract_features).reset_index(drop=True)
feature_df.to_csv("normal_features.csv", index=False)

使用特征列： ['smart_5_raw', 'smart_187_raw', 'smart_188_raw', 'smart_198_raw', 'smart_197_raw', 'smart_241_raw', 'smart_12_raw', 'smart_7_raw', 'smart_4_raw', 'smart_193_raw', 'smart_190_raw']


/tmp/ipykernel_4015391/4005367185.py:68: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  feature_df = df.groupby('serial_number').apply(extract_features).reset_index(drop=True)


In [2]:
len(feature_df)

2322

In [3]:
feature_df.head()

,smart_5_raw_max,smart_5_raw_mean,smart_5_raw_cv,smart_5_raw_diff_mean,smart_5_raw_diff_max,smart_5_raw_diff_std,smart_5_raw_slope,smart_5_raw_skew,smart_5_raw_kurt,smart_5_raw_last,...,smart_190_raw_cv,smart_190_raw_diff_mean,smart_190_raw_diff_max,smart_190_raw_diff_std,smart_190_raw_slope,smart_190_raw_skew,smart_190_raw_kurt,smart_190_raw_last,failure,serial_number
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.018802,0.000000,1.0,0.632456,0.000000,0.374166,-2.800000,29.0,0,ZHZ07RNQ
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,24.0,0,ZHZ085SK
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,20.0,0,ZHZ08A28
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.009727,0.000000,1.0,0.632456,0.000000,-2.645751,7.000000,39.0,0,ZHZ08KC5
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.040393,0.666667,3.0,2.422120,0.666667,-0.620098,-0.809375,39.0,0,ZHZ096WB
